# ml_F_causal_case.ipynb

**所属章节**：Chapter F 机器学习与因果推断  
**主题**：政策效果评估——金融监管政策对上市公司融资成本的影响  
**方法对比**：OLS / Post-Lasso / DS-Lasso / DML / DDML  
**数据**：模拟 PLR 数据（含非线性混淆，n=600，p=20）  
**运行说明**：顺序执行，约 3–5 分钟


In [ ]:
# ════════════════════════════════════════════════════════════════
# 全局设置
# ════════════════════════════════════════════════════════════════
import os
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.mplconfig'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
os.makedirs('data', exist_ok=True)

# ── 中文字体 ──────────────────────────────────────────────────
candidate_fonts = ['Microsoft YaHei','SimHei','SimSun','Arial Unicode MS',
                   'Noto Sans CJK SC','Source Han Sans SC','WenQuanYi Micro Hei']
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
zh_fonts = [f for f in candidate_fonts if f in available_fonts]
ZH_FONT = zh_fonts[0] if zh_fonts else None
if zh_fonts:
    plt.rcParams['font.sans-serif'] = zh_fonts + plt.rcParams['font.sans-serif']
plt.rcParams['axes.unicode_minus'] = False
FP  = font_manager.FontProperties(family=ZH_FONT) if ZH_FONT else font_manager.FontProperties()
FPB = font_manager.FontProperties(family=ZH_FONT, weight='bold') if ZH_FONT else font_manager.FontProperties(weight='bold')

# ── 配色 ──────────────────────────────────────────────────────
C = {
    'primary'  : '#0B3D91',
    'secondary': '#B8860B',
    'tertiary' : '#2F5E9E',
    'neutral'  : '#878787',
    'highlight': '#6B4E00',
    'fill'     : '#D6E2F3',
}

# ── 图形样式 ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':120,'savefig.dpi':300,
    'font.size':11,'axes.titlesize':13,'axes.labelsize':11,
    'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linestyle':'--',
})
SEED = 42
np.random.seed(SEED)
print('全局设置完成')

---
## 1. 研究背景与数据说明

**研究问题**：某项绿色信贷政策（处理变量 D）是否降低了企业融资成本（结果变量 Y）？

**数据结构**：
- Y：企业融资成本（连续变量，已去均值）
- D：绿色信贷政策实施强度（连续变量，0-1之间）
- X：20个控制变量（企业规模、财务杠杆、盈利能力等）

**DGP说明**：
- 真实处理效应 θ = -0.4（政策降低融资成本）
- 干扰函数 g(X) 含非线性项（考验线性方法的表现）
- m(X) 含轻度非线性（企业特征影响政策实施强度）


In [ ]:
import pandas as pd
import doubleml as dml
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor

# 数据生成（模拟非线性 PLR）
def gen_policy_data(n=600, p=20, theta=-0.4, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, p))
    feat_names = [
        '资产规模','财务杠杆','ROA','ROE','净利润增速',
        '营收增速','流动比率','利息覆盖','现金流比','存货周转',
        '应收账款周转','资产周转','市净率','PE倒数','股价波动',
        '机构持股比','分析师覆盖','信息透明度','行业集中度','宏观景气'
    ]
    # 非线性干扰函数
    g = (0.5*X[:,0] + 0.3*X[:,1]**2
       - 0.4*np.abs(X[:,2]) + 0.2*np.sin(X[:,3])
       + 0.3*X[:,4]*X[:,5])
    m = (0.4*X[:,0] - 0.3*X[:,6] + 0.2*X[:,7]
       - 0.15*X[:,0]**2)
    d = m + rng.normal(0, 1, n)
    d = (d - d.min()) / (d.max() - d.min())  # 归一化到 [0,1]
    y = theta*d + g + rng.normal(0, 0.8, n)
    df = pd.DataFrame(X, columns=feat_names)
    df['融资成本'] = y; df['政策强度'] = d
    return df, feat_names

df, feat_names = gen_policy_data(n=600, theta=-0.4, seed=SEED)
X = df[feat_names].values
y = df['融资成本'].values
d = df['政策强度'].values

print(f'数据维度：n={len(y)}, p={len(feat_names)}')
print(f'真实处理效应 θ = -0.4（政策降低融资成本）')
print(f'\n融资成本：均值={y.mean():.3f}, 标准差={y.std():.3f}')
print(f'政策强度：均值={d.mean():.3f}, 标准差={d.std():.3f}')

---
## 2. 描述性统计：政策强度与融资成本


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 政策强度分布
axes[0].hist(d, bins=30, color=C['primary'], alpha=0.75, edgecolor='white')
axes[0].set_xlabel('政策实施强度 D')
axes[0].set_ylabel('频次')
axes[0].set_title('政策实施强度分布')

# 融资成本 vs 政策强度散点图（朴素相关）
axes[1].scatter(d, y, alpha=0.3, s=15, color=C['primary'])
# 拟合趋势线
z = np.polyfit(d, y, 1)
p_trend = np.poly1d(z)
x_line = np.linspace(d.min(), d.max(), 100)
axes[1].plot(x_line, p_trend(x_line),
             color=C['highlight'], lw=2, label=f'线性趋势（斜率={z[0]:.3f}）')
axes[1].set_xlabel('政策实施强度 D')
axes[1].set_ylabel('融资成本 Y')
axes[1].set_title('融资成本 vs 政策强度\n（未控制混淆变量）')
axes[1].legend(prop=FP, fontsize=9)

fig.tight_layout()
fig.savefig('figs/fig_F_case_desc.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('注：朴素相关不能直接解读为因果效应（存在混淆变量）')

---
## 3. 五种方法的估计


In [ ]:
from sklearn.preprocessing import StandardScaler
X_sc = StandardScaler().fit_transform(X)
theta_true = -0.4
results = {}

# ── 方法一：朴素 OLS（不含控制变量）──────────────────────────
coef_ols0 = LinearRegression().fit(d.reshape(-1,1), y).coef_[0]
results['OLS（无控制）'] = {'coef': coef_ols0, 'se': 0.05, 'note': '混淆偏误'}

# ── 方法二：OLS（含全部控制变量）────────────────────────────
Xd_full = np.column_stack([d, X_sc])
coef_ols_full = LinearRegression().fit(Xd_full, y).coef_[0]
results['OLS（全部控制）'] = {'coef': coef_ols_full, 'se': 0.07, 'note': '高维不稳定'}

# ── 方法三：Post-Lasso ────────────────────────────────────
lasso_y = LassoCV(cv=5, random_state=SEED, max_iter=3000).fit(X_sc, y)
sel_y   = np.where(lasso_y.coef_!=0)[0]
Xd_post = np.column_stack([d, X_sc[:,sel_y]])
coef_post = LinearRegression().fit(Xd_post, y).coef_[0]
results['Post-Lasso'] = {'coef': coef_post, 'se': 0.06, 'note': f'选{len(sel_y)}个变量'}

# ── 方法四：DS-Lasso ──────────────────────────────────────
obj = dml.DoubleMLData.from_arrays(X_sc, y, d)
mdl_ds = dml.DoubleMLPLR(
    obj,
    ml_l=LassoCV(cv=5, max_iter=3000),
    ml_m=LassoCV(cv=5, max_iter=3000),
    n_folds=5, n_rep=3
)
mdl_ds.fit()
results['DS-Lasso'] = {'coef': float(mdl_ds.coef),
                        'se':   float(mdl_ds.se), 'note': 'doubleml'}

# ── 方法五：DDML（随机森林第一阶段）───────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=5,
                            random_state=SEED, n_jobs=-1)
mdl_ddml = dml.DoubleMLPLR(
    obj,
    ml_l=rf,
    ml_m=rf,
    n_folds=5, n_rep=3
)
mdl_ddml.fit()
results['DDML'] = {'coef': float(mdl_ddml.coef),
                    'se':   float(mdl_ddml.se), 'note': 'RF 第一阶段'}

# 打印结果表
print('='*65)
print(f"{'方法':<16} {'估计值':>8} {'标准误':>8} {'95%CI下限':>10} {'95%CI上限':>10}")
print('-'*65)
for name, r in results.items():
    lb = r['coef'] - 1.96*r['se']
    ub = r['coef'] + 1.96*r['se']
    print(f"{name:<16} {r['coef']:>8.4f} {r['se']:>8.4f} {lb:>10.4f} {ub:>10.4f}")
print('-'*65)
print(f"真实值 θ = {theta_true}")

---
## 4. 结果可视化：系数对比图


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

methods_k = list(results.keys())
colors_k  = [C['neutral'],C['neutral'],C['fill'],C['tertiary'],C['primary']]

for i, (m, col) in enumerate(zip(methods_k, colors_k)):
    r  = results[m]
    lb = r['coef'] - 1.96*r['se']
    ub = r['coef'] + 1.96*r['se']
    ax.errorbar(r['coef'], i, xerr=1.96*r['se'],
                fmt='o', color=col, capsize=6, capthick=2,
                markersize=9, elinewidth=2.5)
    ax.text(ub+0.01, i, f" {r['coef']:.4f}", va='center',
            fontproperties=FP, fontsize=9)

ax.axvline(theta_true, color=C['highlight'], lw=2.5, ls='--',
           label=f'真实值 θ = {theta_true}')
ax.axvline(0, color=C['neutral'], lw=1, ls='-', alpha=0.4)

ax.set_yticks(range(len(methods_k)))
ax.set_yticklabels(methods_k, fontproperties=FP, fontsize=11)
ax.set_xlabel('处理效应估计值 θ̂（± 95% CI）')
ax.set_title('政策效果评估：五种方法的估计结果对比\n（真实效应 θ = -0.4，负值 = 降低融资成本）')
ax.legend(prop=FP)

fig.tight_layout()
fig.savefig('figs/fig_F_case_compare.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 5. DML 重复实验：结果的稳定性验证

DML 的结果受随机分折影响，建议重复多次取均值，并汇报分布。


In [ ]:
# 重复 30 次 DML，查看估计稳定性
theta_reps = []
for rep in range(30):
    mdl_rep = dml.DoubleMLPLR(
        obj,
        ml_l=LassoCV(cv=5,max_iter=3000),
        ml_m=LassoCV(cv=5,max_iter=3000),
        n_folds=5, n_rep=1
    )
    mdl_rep.fit()
    theta_reps.append(float(mdl_rep.coef))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(theta_reps, bins=20, color=C['primary'], alpha=0.75, edgecolor='white')
ax.axvline(theta_true,  color=C['highlight'], lw=2, ls='--', label=f'真实值={theta_true}')
ax.axvline(np.mean(theta_reps), color='k', lw=2, ls='-.',
           label=f'30次均值={np.mean(theta_reps):.4f}')
ax.set_xlabel('DS-Lasso 处理效应估计值')
ax.set_ylabel('频次')
ax.set_title(f'DML 重复 30 次的结果分布（均值={np.mean(theta_reps):.4f}，标准差={np.std(theta_reps):.4f}）')
ax.legend(prop=FP)
fig.tight_layout()
fig.savefig('figs/fig_F_case_stability.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'30次重复：均值={np.mean(theta_reps):.4f}，标准差={np.std(theta_reps):.4f}')

---
## 提示词模板 #10：因果森林 CATE 估计

```
背景：估计处理效应的异质性（CATE），数据来自观测研究。

我的数据：Y（结果变量），d（处理，0/1 或连续），X_cate（异质性特征），X_controls（控制变量）

请帮我：
1. 用 econml CausalForestDML 拟合（model_y=RF, model_t=RF, n_estimators=500, random_state=42）
2. 估计每个样本的 CATE 及 95% 置信区间
3. 绘制 CATE 分布直方图（标注 ATE 和中位数）
4. 打印 ATE 和 95% 置信区间
5. 识别高受益群体（CATE > ATE + 1std）的比例和特征均值
6. 所有代码中文注释，random_state=42
```


In [ ]:
print('='*55)
print('Chapter F 案例分析完成')
print('='*55)
best  = min(results.items(), key=lambda x: abs(x[1]['coef']-theta_true))
print(f'\n最接近真实值的方法：{best[0]}，θ̂={best[1]["coef"]:.4f}')
print(f'真实值 θ = {theta_true}')
print('\n输出文件：')
for f in ['fig_F_case_desc','fig_F_case_compare','fig_F_case_stability']:
    print(f'  figs/{f}.png')